## 2.1 Spatial Autocorrelation

### 2.1.1 Spatial Autocorrelation

**Spatial autocorrelation (Spatial Dependence)** means that observations at nearby locations are more similar (or more dissimilar) than would be expected under randomness. It violates the assumption of independence in classical statistics.

- **Positive spatial dependence**: Similar values cluster (high–high, low–low).
- **Negative spatial dependence**: Dissimilar values are adjacent (high–low, low–high).
- **No spatial dependence**: Values are randomly distributed in space.

Formally, for variable $z$ at locations $i$ and $j$:

$$\text{Cov}(z_i, z_j) \neq 0 \quad \text{when } i \text{ and } j \text{ are spatially related}$$

### 2.1.2 Spatial Weight Matrix $\mathbf{W}$

Spatial structure is encoded in a **spatial weight matrix** $\mathbf{W}$, where $w_{ij}$ represents the strength of connection between locations $i$ and $j$.

**Common specifications (choose based on data type):**

- **For polygons (areas):**
  - **Binary contiguity**: $w_{ij} = 1$ if $i$ and $j$ share a border (Rook contiguity) or a vertex (Queen contiguity), else $w_{ij} = 0$.
    - *Use when working with areal data (e.g., census tracts, districts).*
  - **Row-standardized**: Each row sums to 1: $\tilde{w}_{ij} = w_{ij} / \sum_j w_{ij}$.
    - *Often applied after constructing a contiguity matrix.*

- **For points (locations):**
  - **Distance-based**: $w_{ij} = 1/d_{ij}^\alpha$ or $w_{ij} = \exp(-\beta d_{ij})$ for distance $d_{ij}$ between $i$ and $j$.
    - *Use for scattered points, especially when proximity matters.*
  - **K-nearest neighbors**: $w_{ij} = 1$ if $j$ is among the $k$ nearest neighbors of $i$, else 0.
    - *Suitable for both point data and irregular polygons when nearest proximity is meaningful.*

**Matrix form:**

$$\mathbf{W} = \begin{pmatrix}
0 & w_{12} & \cdots & w_{1n} \\
w_{21} & 0 & \cdots & w_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
w_{n1} & w_{n2} & \cdots & 0
\end{pmatrix}, \quad w_{ii} = 0$$

### 2.1.3 Spatial Lag

The **spatial** lag of a variable $z$ at location $i$ means that the value at $i$ is influenced by the values of its neighbors. Specifically, the spatial lag is a weighted average: you take each neighboring location $j$, multiply its value $z_j$ by how closely it is connected to $i$ (the weight $w_{ij}$), and sum these up. So, the spatial lag shows how much the surrounding values "pull" or affect the value at location $i$. It's a way to capture and measure the influence of nearby observations in space.

$$(\mathbf{W}\mathbf{z})_i = \sum_{j=1}^{n} w_{ij} \, z_j$$

With row-standardized $\mathbf{W}$, this is the mean of $z$ in the neighborhood of $i$. The full vector is:

$$\mathbf{W}\mathbf{z} = \begin{pmatrix} (\mathbf{W}\mathbf{z})_1 \\ \vdots \\ (\mathbf{W}\mathbf{z})_n \end{pmatrix}$$

Spatial lag is the building block for Moran's I, LISA, and spatial regression (e.g. lag model $y = \rho \mathbf{W}y + \mathbf{X}\beta + \varepsilon$).

### 2.1.4 Global Moran's I

**Moran's I** provides a way to measure the **overall spatial autocorrelation** in a dataset. The formula balances the spatial structure (encoded in the weights $w_{ij}$) with the magnitude and distribution of the variable $z$ across locations.

**Scalar formula:**

$$I = \frac{n}{\sum_i \sum_j w_{ij}} \cdot \frac{\sum_i \sum_j w_{ij} (z_i - \bar{z})(z_j - \bar{z})}{\sum_i (z_i - \bar{z})^2}$$

Here, $n$ is the number of locations, $\bar{z}$ is the mean of $z$, and $w_{ij}$ represents the spatial relationship between locations $i$ and $j$.
- $I > 0$: similar values cluster together (positive spatial autocorrelation)
- $I < 0$: dissimilar values are neighbors (negative spatial autocorrelation)
- $I \approx 0$: no global spatial pattern

**Why such a formula form:**
1) The term $(z_i-\bar z)(z_j-\bar z)$ measures the similarity between neighboring locations:
   - Both values high or both low: product is positive (indicates similarity).
   - One high, one low: product is negative (indicates dissimilarity).
   - Both close to the mean: product is near zero (minimal contribution).
   - This reflects the fundamental idea of "covariance."

2) The term $\sum_i\sum_j w_{ij}(z_i-\bar z)(z_j-\bar z)$ adds spatial weights into calculation:
   - Assign larger weights to location pairs that "should be compared" (neighbors or close points).
   - Assign smaller or even zero weights to pairs that "should not be compared" (far away or not neighbors).
   - So, the numerator is essentially the "weighted sum of neighborhood covariances" (more precisely, it's the weighted sum of cross-products).

3) To make Moran’s I dimensionless and comparable, you need to normalize by a “global scale.” The natural choice is the total sum of squared deviations (global variance): $\sum_i (z_i-\bar z)^2$.

4) Suppose you multiply every weight $w_{ij}$ by 100. The numerator of Moran's I would also be multiplied by 100, so the statistic itself would become much larger—even though the underlying spatial relationships haven't changed. This clearly isn't desirable; it just reflects a change in the units of your weights. So, we need to scale the statistic so that it's not affected by the absolute magnitude of the weights. To do this, we divide by the total sum of all weights: $ S_0 = \sum_i \sum_j w_{ij} $.

5) Finally, use $n$ as normalization factor in the numerator. If we apply row-standardization on spatial weights, where each row of the weight matrix is standardized to sum to 1 (that is, $\sum_j \tilde w_{ij} = 1$ for all $i$), we have $S_0 = \sum_i\sum_j \tilde w_{ij} = n$. In this common case, the normalization factor $\frac{n}{S_0}$ becomes $\frac{n}{n} = 1$. This means that for the most widely-used (row-standardized) weights, multiplying by $n$ exactly cancels the effect of the weights' total sum, and Moran’s I takes on a very natural form: $ I = \frac{\sum_i \sum_j w_{ij} (z_i - \bar{z})(z_j - \bar{z})}{\sum_i (z_i - \bar{z})^2} $
   - In other words, Moran’s I becomes a direct comparison between each observation’s deviation from the mean and the deviation of its neighbors’ weighted average—making the measure highly intuitive. If we did *not* include the $n$ factor, then with row-standardized weights the statistic would be scaled down by $\frac{1}{n}$ and be much smaller, making the interpretation more awkward.

**Matrix form (when working with vectors):**

This formula allows the computation of Moran's I using linear algebra operations, which is practical for large datasets.

$$I = \frac{n}{\mathbf{1}'\mathbf{W}\mathbf{1}} \cdot \frac{\mathbf{z}'\mathbf{M}\mathbf{W}\mathbf{M}\mathbf{z}}{\mathbf{z}'\mathbf{M}\mathbf{z}}, \quad \mathbf{M} = \mathbf{I} - \frac{1}{n}\mathbf{1}\mathbf{1}'$$

First we explain what is centring matrix $\mathbf{M}$. It is a projection matrix to have variable $\mathbf{z}$ remove its mean $\bar z$. As $\mathbf{z}$ is actually made of mean and variable's difference:

$$\mathbf z = \underbrace{\bar z\,\mathbf1}_{\text{direction of mean}} + \underbrace{(\mathbf z-\bar z\,\mathbf1)}_{\text{diff to mean}}$$

The centring matrix is applied to make variable $\mathbf{z}$ calculate its difference with its mean value, which is $\mathbf{x}$:
- $\mathbf{x}=\mathbf{M}\mathbf{z}=\left(\mathbf{I}-\frac{1}{n}\mathbf{1}\mathbf{1}'\right)\mathbf{z}=\mathbf{z}-\frac{1}{n}\mathbf{1}(\mathbf{1}'\mathbf{z})=\mathbf{z}-\bar z\,\mathbf{1}$
- $\mathbf{x}=\mathbf{M}\mathbf{z}=\mathbf{z}-\bar z\,\mathbf{1},\quad\text{which is}\quad x_i=z_i-\bar z$ in the original formula

1) The denominator $\sum_i (z_i-\bar z)^2 = \sum_i x_i^2$ is $\mathbf{x}'\mathbf{x} = (\mathbf{M}\mathbf{z})'(\mathbf{M}\mathbf{z})
= \mathbf{z}'\mathbf{M}'\mathbf{M}\mathbf{z}=\mathbf{z}'\mathbf{M}\mathbf{z}$ in matrix expression. 
2) The numerator $\sum_i\sum_j w_{ij}(z_i-\bar z)(z_j-\bar z)=\sum_i\sum_j w_{ij}x_i x_j$ is $\mathbf{x}'\mathbf{W}\mathbf{x}=(\mathbf{M}\mathbf{z})'\mathbf{W}(\mathbf{M}\mathbf{z})=\mathbf{z}'\mathbf{M}'\mathbf{W}\mathbf{M}\mathbf{z}=\mathbf{z}'\mathbf{M}\mathbf{W}\mathbf{M}\mathbf{z}$ matrix expression.
3) The weight denominator $S_0=\sum_i\sum_j w_{ij}$ is $S_0=\mathbf{1}'\mathbf{W}\mathbf{1}$ in matrix expression.


**Expected value under the null hypothesis (random spatial arrangement):**
$E[I] = -\frac{1}{n-1}$

**Variance** (for significance testing, using $S_1 = \frac{1}{2}\sum_i \sum_j (w_{ij} + w_{ji})^2$, $S_2 = \sum_i \bigl( \sum_j w_{ij} + \sum_j w_{ji} \bigr)^2$):
$\text{Var}(I) = \frac{n^2 S_1 - n S_2 + 3(\mathbf{1}'\mathbf{W}\mathbf{1})^2}{(\mathbf{1}'\mathbf{W}\mathbf{1})^2 (n^2 - 1)} - (E[I])^2$

**Z-score for inference:**
$Z_I = \frac{I - E[I]}{\sqrt{\text{Var}(I)}}$

This allows us to formally test whether the observed spatial autocorrelation is stronger than would be expected by chance.

### 2.1.5 Local Moran's I (LISA)

**LISA** (Local Indicators of Spatial Association) evaluate **local** spatial autocorrelation at each location. **Local Moran's I** for location $i$ is:

$$
I_i = \frac{(z_i - \bar{z})}{s^2} \sum_{j=1}^{n} w_{ij} (z_j - \bar{z})
$$

where $s^2 = \frac{1}{n}\sum_i (z_i - \bar{z})^2$. In deviation form $x_i = z_i - \bar{z}$:

$$
I_i = \frac{n \, x_i \, (\mathbf{W}\mathbf{x})_i}{\sum_k x_k^2}
$$

**Decomposition:** Global Moran's I is proportional to the sum of local Moran's I:
$$
I \propto \sum_i I_i
$$

**LISA quadrants** (with $x_i = z_i - \bar{z}$, spatial lag $x_i^{\text{lag}} = \sum_j w_{ij} x_j$):

| Quadrant | $x_i$ | $x_i^{\text{lag}}$ | Interpretation |
|----------|---------|----------------------|----------------|
| HH | High | High | High surrounded by high |
| LH | Low | High | Low surrounded by high |
| LL | Low | Low | Low surrounded by low |
| HL | High | Low | High surrounded by low |

HH and LL indicate positive local autocorrelation; LH and HL indicate negative (spatial outliers).

### 2.1.6 Other Local Statistics

- **Getis–Ord \(G_i\)**: Measures concentration of high or low values; \(G_i^*\) includes the location itself in the neighborhood.
- **Local Geary \(c_i\)**: Uses squared differences \((z_i - z_j)^2\); different sensitivity to local structure than Moran.

### 2.1.7 Summary of Key Functions and Expressions

| Concept | Formula / function |
|--------|---------------------|
| Spatial lag | \((\mathbf{W}\mathbf{z})_i = \sum_j w_{ij} z_j\) |
| Global Moran's I | \(I = \frac{n}{\mathbf{1}'\mathbf{W}\mathbf{1}} \frac{\mathbf{z}'\mathbf{W}\mathbf{z}}{\mathbf{z}'\mathbf{M}\mathbf{z}}\) |
| Local Moran's I | \(I_i = \frac{n \, x_i \, (\mathbf{W}\mathbf{x})_i}{\sum_k x_k^2},\; x_i = z_i - \bar{z}\) |
| Null expectation | \(E[I] = -\frac{1}{n-1}\) |
| LISA quadrants | HH, LH, LL, HL (by sign of \(x_i\) and spatial lag) |

### 2.1.8 Python Implementation

Use **libpysal** (spatial weights) and **esda** (Moran, LISA):

```bash
pip install libpysal esda geopandas
```

- **Weights**: `libpysal.weights.Queen.from_dataframe(gdf)` or `Rook`, `KNN`, etc.
- **Global Moran**: `esda.moran.Moran(z, w)` → `.I`, `.p_sim`, `.z_sim`
- **Local Moran (LISA)**: `esda.moran.Moran_Local(z, w)` → `.Is`, `.q`, `.p_sim`

In [1]:
# Install if needed: pip install libpysal esda geopandas
import numpy as np
from libpysal.weights import lat2W
from esda.moran import Moran, Moran_Local

# Example: 5x5 grid, n=25
n = 25
np.random.seed(42)
z = np.random.randn(n)  # variable of interest
w = lat2W(5, 5)         # rook weights on 5x5 grid

# Global Moran's I
moran = Moran(z, w)
print(f"Moran's I: {moran.I:.4f}")
print(f"E[I] under H0: {moran.EI:.4f}")
print(f"p-value (permutation): {moran.p_sim:.4f}")
print(f"Z-score: {moran.z_sim:.4f}")

# Local Moran (LISA)
lisa = Moran_Local(z, w)
print(f"\nLocal I (first 5): {lisa.Is[:5]}")
print(f"Quadrants (1=HH,2=LH,3=LL,4=HL): {lisa.q[:5]}")
print(f"LISA p-values (first 5): {lisa.p_sim[:5]}")

/Users/sijieyang/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Moran's I: 0.2424
E[I] under H0: -0.0417
p-value (permutation): 0.0350
Z-score: 1.7769

Local I (first 5): [-0.01637403  0.02955829  0.78097768  0.26701239 -0.09236479]
Quadrants (1=HH,2=LH,3=LL,4=HL): [4 1 1 1 2]
LISA p-values (first 5): [0.479 0.024 0.044 0.34  0.037]


In [2]:
# Spatial lag: W @ z (weighted average of neighbors)
z_dev = z - z.mean()
spatial_lag = w.sparse @ z_dev  # (Wz)_i for each i
# Manual check: I = n * z' W z / (z' z) for row-standardized W
n = len(z)
S0 = w.sparse.sum()  # sum of all w_ij
I_manual = (n / S0) * (z_dev @ spatial_lag) / (z_dev @ z_dev)
print(f"Moran's I (manual): {I_manual:.6f}")
print(f"Moran's I (esda):   {moran.I:.6f}")

Moran's I (manual): 0.242448
Moran's I (esda):   0.242448
